# 3.1.1 — Cartan ilişkilerinin 0-form ve 1-form üzerinde ispatı (operatör-eşitliği yöntemi)

**İddia.** Aşağıdaki altı Cartan ilişkisinin tamamı, $\Omega^\bullet(M)$'nin generator çiftine — bir 0-form $f$ ve bir 1-form $df$'e — uygulandığında kapanır:

$$
\begin{aligned}
(1)\quad & d^2 = 0 \\
(2)\quad & d\,\mathcal{L}_X - \mathcal{L}_X\,d = 0 \\
(3)\quad & d\,\iota_X + \iota_X\,d = \mathcal{L}_X \\
(4)\quad & \mathcal{L}_X\mathcal{L}_Y - \mathcal{L}_Y\mathcal{L}_X = \mathcal{L}_{[X,Y]} \\
(5)\quad & \mathcal{L}_X\iota_Y - \iota_Y\mathcal{L}_X = \iota_{[X,Y]} \\
(6)\quad & \iota_X\iota_Y + \iota_Y\iota_X = 0
\end{aligned}
$$

**Strateji.** Her bir özdeşlik bir *operator equation* (operatör eşitliği) olarak ifade edilir. `AgreementOnGenerators` stratejisi: iki graded derivation $\Omega^\bullet(M)$'nin üreteç kümesi üzerinde aynı değerleri veriyorsa, wedge ve $d$ altında kapalı olan tüm cebir üzerinde de aynı değeri verir. Üreteç kümesi $\{f, df\}$ — yani **0-form $f$ ve 1-form $df$**. İspat: her üreteç için iki tarafın `ExpandAndSimplify` ile aynı normal forma indiğini göster.

In [1]:
try:
    import jacopy  # noqa: F401
except ModuleNotFoundError:
    import sys
    from pathlib import Path
    here = Path.cwd().resolve()
    for candidate in (here, *here.parents):
        if (candidate / "jacopy" / "__init__.py").is_file():
            sys.path.insert(0, str(candidate))
            break
    import jacopy  # noqa: F401

from jacopy.algebra.derivation import Act, Derivation, compose
from jacopy.brackets.lie import LieBracket
from jacopy.calculus.cartan import CartanCalculus, RELATIONS
from jacopy.calculus.exterior_algebra import ExteriorAlgebra
from jacopy.calculus.exterior_d import d
from jacopy.calculus.interior import interior
from jacopy.calculus.lie_derivative import lie_derivative
from jacopy.core.expr import Integer, Sum, Symbol
from jacopy.core.properties import Graded
from jacopy.core.registry import PropertyRegistry
from jacopy.proof.expansion import default_engine
from jacopy.proof.verifier import prove_operator_equation

## 1. Kurulum — algebra, vektör alanları, Cartan paketi

* `f`: 0-form üreteci → `Graded(degree=0)`.
* `ExteriorAlgebra((f,))` üreteçleri: `(f, df)` — yani **0-form + 1-form** çifti.
* `X, Y`: derece-0 derivation'lar (vektör alanları).
* Cartan-mode $\mathcal{L}_X$: paket içinde $d\circ\iota_X + \iota_X\circ d$ olarak tanımlıdır (relation (3) bunun *tanım gereği* sağlandığı anlamına gelir).
* Vektör-alanı bracket'i: `LieBracket()` — kompozisyon-tabanlı $[X, Y] = X\cdot Y - Y\cdot X$.

In [2]:
reg = PropertyRegistry()

f = Symbol("f")
reg.declare(f, Graded(degree=0))
algebra = ExteriorAlgebra((f,))

X = Derivation("X", degree=0)
Y = Derivation("Y", degree=0)

lie = lambda V: lie_derivative(V, definition="cartan")
iota = interior

cartan = CartanCalculus(
    d=d,
    lie_derivative=lie,
    interior=iota,
    vector_bracket=LieBracket(),
)

engine = default_engine(registry=reg)

print("Üreteçler (algebra.generators):")
for g in algebra.generators:
    prop = reg.get(g, Graded)
    deg = prop.degree if prop is not None else "(üretilen)"
    print(f"  {g}   (derece {deg})")
print(f"\nVektör alanları : X (deg 0), Y (deg 0)")
print(f"Lie-mode        : cartan (yani L_X := d∘ι_X + ι_X∘d)")
print(f"VF bracket      : {cartan.vector_bracket.name}")


Üreteçler (algebra.generators):
  f   (derece 0)
  d(f)   (derece (üretilen))

Vektör alanları : X (deg 0), Y (deg 0)
Lie-mode        : cartan (yani L_X := d∘ι_X + ι_X∘d)
VF bracket      : [·,·]


## 2. Görselleştirme yardımcısı

Her ilişki için iki taraf arasındaki eşitliği üreteç bazında göstermek için küçük bir yardımcı: bir operatörü bir üretece uygulayıp engine + simplify ile normal forma indirir.

In [3]:
from jacopy.algorithms.simplify import simplify
from jacopy.algorithms.product_rule import product_rule

def normal_form(op_expr, generator):
    """Operatör ifadesini bir üretece uygulayıp engine fix-point'e indir."""
    current = Act(op_expr, generator)
    for _ in range(40):
        expanded, _ = engine.expand(current)
        after = simplify(product_rule(expanded, reg), reg)
        if after == current:
            return current
        current = after
    return current


def show_relation(name, lhs_op, rhs_op):
    """İlişkiyi her iki üreteç üzerinde aç ve eşitliği göster."""
    print(f"--- {name} ---")
    for g in algebra.generators:
        lhs_val = normal_form(lhs_op, g)
        rhs_val = normal_form(rhs_op, g)
        eq = "✓" if lhs_val == rhs_val else "✗"
        print(f"  {eq}  generator {g!s:<6}  LHS({g}) = {lhs_val}")
        print(f"        {' '*len(str(g)):<6}  RHS({g}) = {rhs_val}")
    print()


def show_branches(chain, *, label=""):
    """AgreementOnGenerators chain'inin her üreteç dalını adım-adım yazdır."""
    parent = chain.steps[0]
    if label:
        print(f"### {label} — alt-adımlar (AgreementOnGenerators dalları)")
    for branch in parent.children:
        print(f"\n  ┌── üreteç dalı: {branch.before} ⟶ {branch.after}")
        print(f"  │   ({len(branch.children)} alt-adım)")
        for i, sub in enumerate(branch.children):
            print(f"  │  [{i}] {sub.rule}")
            print(f"  │       {sub.before}  →  {sub.after}")
        print(f"  └──")


## 3. İlişki (1) — $d^2 = 0$

**Operatör eşitliği:** $d \circ d \equiv 0$ (her form üzerinde).

**Üreteç bazında:**
* $d^2(f) = d(df) \xrightarrow{d^2 = 0\text{ axiom}} 0$
* $d^2(df) = d(d(df)) \xrightarrow{d^2 = 0\text{ axiom}} 0$

Engine'de `DSquaredZeroDefinition` axiom'u `Act(d, Act(d, x))` shape'ini doğrudan $0$'a indirir.

In [4]:
chain1 = cartan.verify("d_squared_zero", algebra=algebra, registry=reg)
print(f"verify(d_squared_zero)  →  KAPANDI ({len(chain1)} adım, kök kuralı: {chain1.steps[0].rule!r})")
print()
show_relation("(1) d² = 0", lhs_op=compose(d, d), rhs_op=Integer(0))
show_branches(chain1, label="(1) d² = 0")


verify(d_squared_zero)  →  KAPANDI (1 adım, kök kuralı: 'AgreementOnGenerators')

--- (1) d² = 0 ---
  ✓  generator f       LHS(f) = 0
                RHS(f) = 0
  ✓  generator d(f)    LHS(d(f)) = 0
                RHS(d(f)) = 0

### (1) d² = 0 — alt-adımlar (AgreementOnGenerators dalları)

  ┌── üreteç dalı: (d * d)(f) ⟶ 0(f)
  │   (3 alt-adım)
  │  [0] product-rule
  │       ((d * d)(f) + (-0(f)))  →  (d(d(f)) + (-0))
  │  [1] d² = 0
  │       d(d(f))  →  0
  │  [2] simplify
  │       (0 + (-0))  →  0
  └──

  ┌── üreteç dalı: (d * d)(d(f)) ⟶ 0(d(f))
  │   (4 alt-adım)
  │  [0] product-rule
  │       ((d * d)(d(f)) + (-0(d(f))))  →  (d(d(d(f))) + (-0))
  │  [1] d² = 0
  │       d(d(f))  →  0
  │  [2] product-rule
  │       (d(0) + (-0))  →  (0 + (-0))
  │  [3] simplify
  │       (0 + (-0))  →  0
  └──


## 4. İlişki (2) — $d\mathcal{L}_X - \mathcal{L}_X d = 0$

**Operatör eşitliği:** $[d, \mathcal{L}_X] \equiv 0$ — yani $d$ ve $\mathcal{L}_X$ değişmeli.

**Üreteç bazında:**
* $f$ üzerinde: $d\mathcal{L}_X(f) - \mathcal{L}_X(df) = d(X(f)) - d(X(f)) = 0$ (Cartan magic + $d^2 = 0$).
* $df$ üzerinde: $d\mathcal{L}_X(df) - \mathcal{L}_X(d(df)) = d\mathcal{L}_X(df) - 0 = d\mathcal{L}_X(df)$; Cartan magic ile $\mathcal{L}_X(df) = d\iota_X(df) + \iota_X(d^2 f) = d(X(f))$ olur, dış $d$ uygulandığında $d^2(X(f)) = 0$.

In [5]:
L_X = lie(X)
chain2 = cartan.verify("d_lie", algebra=algebra, X=X, registry=reg)
print(f"verify(d_lie)  →  KAPANDI ({len(chain2)} adım, kök kuralı: {chain2.steps[0].rule!r})")
print()
show_relation(
    "(2) [d, L_X] = 0",
    lhs_op=Sum(compose(d, L_X), -compose(L_X, d)),
    rhs_op=Integer(0),
)
show_branches(chain2, label="(2) [d, L_X] = 0")


verify(d_lie)  →  KAPANDI (1 adım, kök kuralı: 'AgreementOnGenerators')

--- (2) [d, L_X] = 0 ---
  ✓  generator f       LHS(f) = 0
                RHS(f) = 0
  ✓  generator d(f)    LHS(d(f)) = 0
                RHS(d(f)) = 0

### (2) [d, L_X] = 0 — alt-adımlar (AgreementOnGenerators dalları)

  ┌── üreteç dalı: ((d * L_X) + (-(L_X * d)))(f) ⟶ 0(f)
  │   (12 alt-adım)
  │  [0] Act linearity: (A + B)(x) = A(x) + B(x)
  │       ((d * L_X) + (-(L_X * d)))(f)  →  ((d * L_X)(f) + (-(L_X * d))(f))
  │  [1] product-rule
  │       (((d * L_X)(f) + (-(L_X * d))(f)) + (-0(f)))  →  ((d(L_X(f)) + (-L_X(d(f)))) + (-0))
  │  [2] L_X := d∘ι_X + ι_X∘d (Cartan definition)
  │       L_X(f)  →  ((d * ι_X)(f) + (ι_X * d)(f))
  │  [3] L_X := d∘ι_X + ι_X∘d (Cartan definition)
  │       L_X(d(f))  →  ((d * ι_X)(d(f)) + (ι_X * d)(d(f)))
  │  [4] product-rule
  │       ((d(((d * ι_X)(f) + (ι_X * d)(f))) + (-((d * ι_X)(d(f)) + (ι_X * d)(d(f))))) + (-0))  →  (((d(d(ι_X(f))) + d(ι_X(d(f)))) + (-(d(ι_X(d(f))) + ι_

## 5. İlişki (3) — $d\iota_X + \iota_X d = \mathcal{L}_X$ (Cartan magic)

**Operatör eşitliği:** $\mathcal{L}_X \equiv d\iota_X + \iota_X d$.

Cartan-mode $\mathcal{L}_X$ paket içinde **tam olarak bu formülle tanımlıdır** — yani relation (3) `LieDerivativeCartanDefinition` axiom'unun ifadesidir.

**Üreteç bazında:**
* $f$ üzerinde: $d(\iota_X f) + \iota_X(df) = d(0) + X(f) = X(f) = \mathcal{L}_X(f)$.
* $df$ üzerinde: $d(\iota_X(df)) + \iota_X(d^2 f) = d(X(f)) + 0 = d(X(f)) = \mathcal{L}_X(df)$.

In [6]:
iota_X = iota(X)
chain3 = cartan.verify("cartan_magic", algebra=algebra, X=X, registry=reg)
print(f"verify(cartan_magic)  →  KAPANDI ({len(chain3)} adım, kök kuralı: {chain3.steps[0].rule!r})")
print()
show_relation(
    "(3) dι_X + ι_X d = L_X",
    lhs_op=Sum(compose(d, iota_X), compose(iota_X, d)),
    rhs_op=L_X,
)
show_branches(chain3, label="(3) dι_X + ι_X d = L_X")


verify(cartan_magic)  →  KAPANDI (1 adım, kök kuralı: 'AgreementOnGenerators')

--- (3) dι_X + ι_X d = L_X ---
  ✓  generator f       LHS(f) = X(f)
                RHS(f) = X(f)
  ✓  generator d(f)    LHS(d(f)) = d(X(f))
                RHS(d(f)) = d(X(f))

### (3) dι_X + ι_X d = L_X — alt-adımlar (AgreementOnGenerators dalları)

  ┌── üreteç dalı: ((d * ι_X) + (ι_X * d))(f) ⟶ L_X(f)
  │   (9 alt-adım)
  │  [0] Act linearity: (A + B)(x) = A(x) + B(x)
  │       ((d * ι_X) + (ι_X * d))(f)  →  ((d * ι_X)(f) + (ι_X * d)(f))
  │  [1] L_X := d∘ι_X + ι_X∘d (Cartan definition)
  │       L_X(f)  →  ((d * ι_X)(f) + (ι_X * d)(f))
  │  [2] product-rule
  │       (((d * ι_X)(f) + (ι_X * d)(f)) + (-((d * ι_X)(f) + (ι_X * d)(f))))  →  ((d(ι_X(f)) + ι_X(d(f))) + (-(d(ι_X(f)) + ι_X(d(f)))))
  │  [3] ι_X(f) = 0 on 0-forms
  │       ι_X(f)  →  0
  │  [4] ι_X(df) = X(f)
  │       ι_X(d(f))  →  X(f)
  │  [5] ι_X(f) = 0 on 0-forms
  │       ι_X(f)  →  0
  │  [6] ι_X(df) = X(f)
  │       ι_X(d(f))  →  X(f)
 

## 6. İlişki (4) — $\mathcal{L}_X\mathcal{L}_Y - \mathcal{L}_Y\mathcal{L}_X = \mathcal{L}_{[X,Y]}$

**Operatör eşitliği:** $[\mathcal{L}_X, \mathcal{L}_Y] \equiv \mathcal{L}_{[X,Y]}$.

$[X, Y]$ vektör-alanı bracket'i `LieBracket` ile $X\cdot Y - Y\cdot X$ kompozisyonu. 

**Üreteç bazında:**
* $f$ üzerinde: $\mathcal{L}_X\mathcal{L}_Y(f) - \mathcal{L}_Y\mathcal{L}_X(f) = X(Y(f)) - Y(X(f)) = [X,Y](f) = \mathcal{L}_{[X,Y]}(f)$.
* $df$ üzerinde: Cartan magic'i her iki tarafta açıp $d^2 = 0$ + $\iota_X(df) = X(f)$ ile sadeleştir.

In [7]:
L_Y = lie(Y)
XY = LieBracket().expand(X, Y)
L_XY = lie(XY)
chain4 = cartan.verify("lie_lie", algebra=algebra, X=X, Y=Y, registry=reg)
print(f"verify(lie_lie)  →  KAPANDI ({len(chain4)} adım, kök kuralı: {chain4.steps[0].rule!r})")
print()
show_relation(
    "(4) [L_X, L_Y] = L_[X,Y]",
    lhs_op=Sum(compose(L_X, L_Y), -compose(L_Y, L_X)),
    rhs_op=L_XY,
)
show_branches(chain4, label="(4) [L_X, L_Y] = L_[X,Y]")


verify(lie_lie)  →  KAPANDI (1 adım, kök kuralı: 'AgreementOnGenerators')

--- (4) [L_X, L_Y] = L_[X,Y] ---
  ✓  generator f       LHS(f) = (X(Y(f)) + (-Y(X(f))))
                RHS(f) = (X(Y(f)) + (-Y(X(f))))
  ✓  generator d(f)    LHS(d(f)) = (d(X(Y(f))) + (-d(Y(X(f)))))
                RHS(d(f)) = (d(X(Y(f))) + (-d(Y(X(f)))))

### (4) [L_X, L_Y] = L_[X,Y] — alt-adımlar (AgreementOnGenerators dalları)

  ┌── üreteç dalı: ((L_X * L_Y) + (-(L_Y * L_X)))(f) ⟶ L_((X * Y) + (-(Y * X)))(f)
  │   (31 alt-adım)
  │  [0] Act linearity: (A + B)(x) = A(x) + B(x)
  │       ((L_X * L_Y) + (-(L_Y * L_X)))(f)  →  ((L_X * L_Y)(f) + (-(L_Y * L_X))(f))
  │  [1] L_X := d∘ι_X + ι_X∘d (Cartan definition)
  │       L_((X * Y) + (-(Y * X)))(f)  →  ((d * ι_((X * Y) + (-(Y * X))))(f) + (ι_((X * Y) + (-(Y * X))) * d)(f))
  │  [2] product-rule
  │       (((L_X * L_Y)(f) + (-(L_Y * L_X))(f)) + (-((d * ι_((X * Y) + (-(Y * X))))(f) + (ι_((X * Y) + (-(Y * X))) * d)(f))))  →  ((L_X(L_Y(f)) + (-L_Y(L_X(f)))) + (-(d

## 7. İlişki (5) — $\mathcal{L}_X\iota_Y - \iota_Y\mathcal{L}_X = \iota_{[X,Y]}$

**Operatör eşitliği:** $[\mathcal{L}_X, \iota_Y] \equiv \iota_{[X,Y]}$.

**Üreteç bazında:**
* $f$ üzerinde: $\mathcal{L}_X(\iota_Y f) - \iota_Y(\mathcal{L}_X f) = \mathcal{L}_X(0) - \iota_Y(X(f)) = 0 - 0 = 0$; ayrıca $\iota_{[X,Y]}(f) = 0$ (interior 0-formda sıfır).
* $df$ üzerinde: $\mathcal{L}_X(\iota_Y(df)) - \iota_Y(\mathcal{L}_X(df)) = \mathcal{L}_X(Y(f)) - \iota_Y(d(X(f))) = X(Y(f)) - Y(X(f)) = [X,Y](f) = \iota_{[X,Y]}(df)$.

In [8]:
iota_Y = iota(Y)
iota_XY = iota(XY)
chain5 = cartan.verify("lie_iota", algebra=algebra, X=X, Y=Y, registry=reg)
print(f"verify(lie_iota)  →  KAPANDI ({len(chain5)} adım, kök kuralı: {chain5.steps[0].rule!r})")
print()
show_relation(
    "(5) [L_X, ι_Y] = ι_[X,Y]",
    lhs_op=Sum(compose(L_X, iota_Y), -compose(iota_Y, L_X)),
    rhs_op=iota_XY,
)
show_branches(chain5, label="(5) [L_X, ι_Y] = ι_[X,Y]")


verify(lie_iota)  →  KAPANDI (1 adım, kök kuralı: 'AgreementOnGenerators')

--- (5) [L_X, ι_Y] = ι_[X,Y] ---
  ✓  generator f       LHS(f) = 0
                RHS(f) = 0
  ✓  generator d(f)    LHS(d(f)) = (X(Y(f)) + (-Y(X(f))))
                RHS(d(f)) = (X(Y(f)) + (-Y(X(f))))

### (5) [L_X, ι_Y] = ι_[X,Y] — alt-adımlar (AgreementOnGenerators dalları)

  ┌── üreteç dalı: ((L_X * ι_Y) + (-(ι_Y * L_X)))(f) ⟶ ι_((X * Y) + (-(Y * X)))(f)
  │   (13 alt-adım)
  │  [0] Act linearity: (A + B)(x) = A(x) + B(x)
  │       ((L_X * ι_Y) + (-(ι_Y * L_X)))(f)  →  ((L_X * ι_Y)(f) + (-(ι_Y * L_X))(f))
  │  [1] ι_X(f) = 0 on 0-forms
  │       ι_((X * Y) + (-(Y * X)))(f)  →  0
  │  [2] product-rule
  │       (((L_X * ι_Y)(f) + (-(ι_Y * L_X))(f)) + (-0))  →  ((L_X(ι_Y(f)) + (-ι_Y(L_X(f)))) + (-0))
  │  [3] ι_X(f) = 0 on 0-forms
  │       ι_Y(f)  →  0
  │  [4] L_X := d∘ι_X + ι_X∘d (Cartan definition)
  │       L_X(0)  →  ((d * ι_X)(0) + (ι_X * d)(0))
  │  [5] L_X := d∘ι_X + ι_X∘d (Cartan definition)
  │  

## 8. İlişki (6) — $\iota_X\iota_Y + \iota_Y\iota_X = 0$

**Operatör eşitliği:** İnterior product graded skew (derece $-1$, dolayısıyla iki interior anti-commute).

Bu ilişki `CartanCalculus.RELATIONS`'da yok — `prove_operator_equation` ile doğrudan `AgreementOnGenerators` kullanarak ispatlayacağız.

**Üreteç bazında:**
* $f$ üzerinde: $\iota_X(\iota_Y f) + \iota_Y(\iota_X f) = \iota_X(0) + \iota_Y(0) = 0$ (interior 0-formda sıfır).
* $df$ üzerinde: $\iota_X(\iota_Y(df)) + \iota_Y(\iota_X(df)) = \iota_X(Y(f)) + \iota_Y(X(f)) = 0 + 0 = 0$ ($Y(f), X(f)$ birer 0-form).

Yani her iki üreteç üzerinde her iki taraf da $0$'a iniyor.

In [9]:
lhs6 = Sum(compose(iota_X, iota_Y), compose(iota_Y, iota_X))
rhs6 = Integer(0)

chain6 = prove_operator_equation(
    lhs6, rhs6, algebra,
    registry=reg,
    engine=engine,
)
print(f"prove_operator_equation(ι_X∘ι_Y + ι_Y∘ι_X = 0)  →  KAPANDI ({len(chain6)} adım, kök kuralı: {chain6.steps[0].rule!r})")
print()
show_relation("(6) ι_X∘ι_Y + ι_Y∘ι_X = 0", lhs_op=lhs6, rhs_op=rhs6)
show_branches(chain6, label="(6) ι_X∘ι_Y + ι_Y∘ι_X = 0")


prove_operator_equation(ι_X∘ι_Y + ι_Y∘ι_X = 0)  →  KAPANDI (1 adım, kök kuralı: 'AgreementOnGenerators')

--- (6) ι_X∘ι_Y + ι_Y∘ι_X = 0 ---
  ✓  generator f       LHS(f) = 0
                RHS(f) = 0
  ✓  generator d(f)    LHS(d(f)) = 0
                RHS(d(f)) = 0

### (6) ι_X∘ι_Y + ι_Y∘ι_X = 0 — alt-adımlar (AgreementOnGenerators dalları)

  ┌── üreteç dalı: ((ι_X * ι_Y) + (ι_Y * ι_X))(f) ⟶ 0(f)
  │   (7 alt-adım)
  │  [0] Act linearity: (A + B)(x) = A(x) + B(x)
  │       ((ι_X * ι_Y) + (ι_Y * ι_X))(f)  →  ((ι_X * ι_Y)(f) + (ι_Y * ι_X)(f))
  │  [1] product-rule
  │       (((ι_X * ι_Y)(f) + (ι_Y * ι_X)(f)) + (-0(f)))  →  ((ι_X(ι_Y(f)) + ι_Y(ι_X(f))) + (-0))
  │  [2] ι_X(f) = 0 on 0-forms
  │       ι_Y(f)  →  0
  │  [3] ι_X(f) = 0 on 0-forms
  │       ι_X(0)  →  0
  │  [4] ι_X(f) = 0 on 0-forms
  │       ι_X(f)  →  0
  │  [5] ι_X(f) = 0 on 0-forms
  │       ι_Y(0)  →  0
  │  [6] simplify
  │       ((0 + 0) + (-0))  →  0
  └──

  ┌── üreteç dalı: ((ι_X * ι_Y) + (ι_Y * ι_X))(d(f)) ⟶ 0(

## 9. Tek bir taramada doğrulama — `verify_all`

(1)-(5) ilişkileri tek bir çağrı ile sweep'lenebilir; (6)'yı yukarıda ayrı ele aldık.

In [10]:
results = cartan.verify_all(algebra=algebra, X=X, Y=Y, registry=reg)
print("verify_all sonuçları:")
for name, chain in results.items():
    parent = chain.steps[0]
    n_children = len(parent.children) if parent.children else 0
    print(f"  {name:<18}  {len(chain)} adım, {n_children} alt-üreteç dalı, kök: {parent.rule!r}")
print(f"\nİlişki (6) ayrıca: {len(chain6)} adım, kök: {chain6.steps[0].rule!r}")
print(f"\nToplam 6/6 ilişki kapandı.")

verify_all sonuçları:
  d_squared_zero      1 adım, 2 alt-üreteç dalı, kök: 'AgreementOnGenerators'
  cartan_magic        1 adım, 2 alt-üreteç dalı, kök: 'AgreementOnGenerators'
  d_lie               1 adım, 2 alt-üreteç dalı, kök: 'AgreementOnGenerators'
  lie_lie             1 adım, 2 alt-üreteç dalı, kök: 'AgreementOnGenerators'
  lie_iota            1 adım, 2 alt-üreteç dalı, kök: 'AgreementOnGenerators'

İlişki (6) ayrıca: 1 adım, kök: 'AgreementOnGenerators'

Toplam 6/6 ilişki kapandı.


## 10. Bir adımı LaTeX ile inceleyelim — Cartan magic ($f$ dalı)

`AgreementOnGenerators` her üreteç için ayrı bir alt-zincir üretir. İlişki (3) için $f$ üzerindeki dalı render edelim.

In [11]:
from jacopy.display.jupyter import display_step

# Bonus: Cartan magic'in (3) f-dalı için LaTeX renderini de gösterelim.
parent = chain3.steps[0]
f_branch = parent.children[0]
print(f"LaTeX render — Cartan magic, f dalı:")
display_step(f_branch)


LaTeX render — Cartan magic, f dalı:


\begin{align*}
\left(d \, \iota_X + \iota_X \, d\right)\!\left(f\right) &\to L_X\!\left(f\right) && \text{[check-on-generator]}\;\text{--- generator g = f}
\end{align*}

## Sonuç

$$
\boxed{\text{(1)-(6) altılısı, } \Omega^\bullet(M)\text{'nin generator çifti } \{f, df\}\text{ üzerinde } \texttt{AgreementOnGenerators} \text{ ile kapanır.}}
$$

Önemli noktalar:

* `ExteriorAlgebra((f,))` üreteç kümesi $\{f, df\}$'tir — yani **bir 0-form ve bir 1-form**. Her ilişki bu iki üreteç üzerinde ayrı ayrı ispatlanır; `AgreementOnGenerators` graded derivation'ların üretici kümede eşitliğinden tüm cebir üzerindeki eşitliği çıkarır.
* (3) Cartan magic *tanım gereği* doğru — paket içinde cartan-mode $\mathcal{L}_X$ tam olarak $d\iota_X + \iota_X d$ olarak kurulur.
* (1), (4), (6) `default_engine`'in axiom set'inde bulunan: $d^2 = 0$, $\iota_X^2 = 0$, $\iota_X(f) = 0$, $\iota_X(df) = X(f)$ kuralları üzerine kuruludur.
* (2), (5) iki tanımın bir araya gelmesinden çıkar: $\mathcal{L}_X$ Cartan tanımı + $d^2 = 0$ + $\iota_X$-action kuralları.
* (6) `RELATIONS` listesinde olmadığı için `prove_operator_equation` ile aynı stratejiye doğrudan başvuruldu.

Bu defter Cartan calculus'ün operator-level kapanışını **explicit olarak** belgeler — her iki üretecin (0-form, 1-form) çıktısı `✓` ile işaretlenir.